# Step 2: Feature engineering

Builds the model features from the step-1 tables in `processed/`. Each section computes one group of features for every exposure (all splits), saves it as its own file in `features/`, and frees memory before the next section. Every file keeps `sid` and `wid` in the row order of `exposures.parquet`, so the files can be joined side by side.

History features follow each action type's logging window: clicks are kept for 7 days, carts for 90 days and orders for 365 days before the search. Other users' actions are counted only from before 17 Oct, the day of every search.

Only Section 6 uses the outcomes of the searches being ranked (the labels), and only from training searches, out-of-fold. Those searches are on the same day as the ones being scored, so the main models leave Section 6 out (see Section 6).

| Section | File | Features | Built from | In main models |
|---|---|---|---|---|
| 1. User × product | `section1_user_product` | #1–6 | the user's own history | yes |
| 2. User × shop / brand / category | `section2_affinity` | #7–17 | the user's own history, product attributes | yes |
| 3. Popularity from histories | `section3_popularity` | #18–24 | all users' history before 17 Oct | yes |
| 4. Search-level context | `section4_context` | — | the user's own history, query | no: same value for the whole list (diagnostics) |
| 5. Query × title | `section5_text` | #25–28 | query and title text | yes |
| 6. Same-day outcome statistics | `section6_label_rates` | #29–36 | outcomes of other training searches on 17 Oct | no: same-day information (step 4 check only) |
| 7. Single-feature checks | `single_feature_check.csv` | — | — | — |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, json, psutil
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

ROOT = "/content/drive/MyDrive/BT4222_Group10_JDSearch"
OUT  = f"{ROOT}/processed"                 # step-1 tables
FEAT = f"{ROOT}/features"                  # step-2 output: one file per section
os.makedirs(FEAT, exist_ok=True)

meta = json.load(open(f"{OUT}/meta.json"))
SEED, DAY, SEP = meta["seed"], 86_400, "\x18"
TYPE  = meta["type_codes"]     # history action types (NOT labels): {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
LABEL = meta["label_codes"]    # exposure labels: {"none": 0, "click": 1, "cart": 2, "order": 3}

def ram(tag=""):
    print(f"[{tag}] RAM in use: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

def pair_key(a, b):
    """Two id columns -> one int64 number per row, so (search, product) or (search, shop) pairs match fast."""
    return (np.asarray(a, dtype=np.int64) << 32) | np.asarray(b, dtype=np.int64)

def save_section(df, name):
    """Save one section's features. Rows follow exposures.parquet; sid and wid are kept for alignment checks."""
    df.to_parquet(f"{FEAT}/{name}.parquet", index=False, compression="zstd")
    print(f"saved {name}: {df.shape[1] - 2} features x {len(df):,} rows")

_tok_cache = {}
def tokens(text):
    """Set of distinct token IDs in an anonymised text; cached because the same query appears many times."""
    if not isinstance(text, str):
        return frozenset()
    t = _tok_cache.get(text)
    if t is None:
        t = _tok_cache[text] = frozenset(text.split(SEP)) - {""}
    return t

ex = pd.read_parquet(f"{OUT}/exposures.parquet")      # sid, wid, label, split: the row order every family follows
TR = (ex.split == 0).to_numpy()                       # training rows: the only labels any feature may use
print(f"exposures {len(ex):,} | training rows {TR.sum():,} | seed {SEED}")
ram("setup")

Mounted at /content/drive
exposures 15,510,012 | training rows 10,743,610 | seed 4222
[setup] RAM in use: 0.6 GB


## Section 1: User × product (#1–6)

Has the user acted on this exact product before, and how often within each stream's window: actions (clicks, carts and orders) in 7 days, carts in 90 days, orders in 365 days? It also records how many days ago the most recent action and the most recent order happened. Counts are 0, and recency is NaN, when there is no such action.


In [ ]:
hist = pd.read_parquet(f"{OUT}/history.parquet", columns=["sid", "wid", "type", "age_s"])   # kept for Sections 2-4
ex_key, h_key = pair_key(ex.sid, ex.wid), pair_key(hist.sid, hist.wid)

section1 = ex[["sid", "wid"]].copy()
section1["item_in_history"] = np.isin(ex_key, np.unique(h_key)).astype("int8")     # touched this product at any time

streams1 = {                                                 # name: (which actions, window in days)
    "action_7d": (hist.type.isin([TYPE["CLICK"], TYPE["CART"], TYPE["ORD"]]), 7),
    "cart_90d":  (hist.type == TYPE["CART"], 90),
    "order_1y":  (hist.type == TYPE["ORD"], 365),
}
for name, (type_mask, days) in streams1.items():
    m = (type_mask & (hist.age_s < days * DAY)).to_numpy()
    keys = pd.Series(h_key[m])                                               # (search, product) of each action in the window
    section1[f"item_{name}_count"] = keys.value_counts().reindex(ex_key, fill_value=0).to_numpy(dtype="int16")
    if name != "cart_90d":                                                   # recency for actions and orders
        youngest = pd.Series(hist.age_s.to_numpy()[m]).groupby(keys.to_numpy()).min()
        section1[f"days_since_item_{name}"] = (youngest.reindex(ex_key).to_numpy() / DAY).astype("float32")
    del m, keys

save_section(section1, "section1_user_product")
chk = section1[TR]
print("check (training rows):", int(chk.item_in_history.sum()), "in history |",
      int(chk.days_since_item_order_1y.isna().sum()), "/", int(chk.days_since_item_action_7d.isna().sum()), "missing recency")
del section1, chk, ex_key, h_key, streams1, type_mask, youngest; gc.collect(); ram("section 1")

saved section1_user_product: 6 features x 15,510,012 rows
check (training rows): 83793 in history | 10734397 / 10679886 missing recency
[section 1] RAM in use: 1.8 GB


## Section 2: User × shop, brand and category (#7–17)

For each stream (actions in 7 days, carts in 90 days, orders in 365 days), the share of the user's actions whose shop, brand or finest category (level 4) matches the candidate's: 9 features. Two query-matched shares, for category and shop, use only last-7-day actions that came from a past query similar to today's (token Jaccard ≥ 0.5).

A share is NaN when the candidate's attribute is unknown, or when the user has no actions with a known attribute in that stream. This keeps missing data separate from a true 0.


In [ ]:
prods = pd.read_parquet(f"{OUT}/products.parquet", columns=["wid", "shop_id", "brand_id", "cate_id_4"]).set_index("wid")
ATTRS = {"shop": "shop_id", "brand": "brand_id", "cate4": "cate_id_4"}
cand_attr = prods.reindex(ex.wid.to_numpy())          # shop / brand / category of every shown product (NaN = unknown)
hist_attr = prods.reindex(hist.wid.to_numpy())        # shop / brand / category of every past action

def share_feature(h_sid, h_val, c_sid, c_val):
    """Share of a user's past actions (with a known value) that match the candidate's value.
    NaN if the candidate's value is unknown or the user has no usable actions."""
    known = ~np.isnan(h_val)
    counts = pd.Series(pair_key(h_sid[known], h_val[known])).value_counts()   # actions per (search, value)
    totals = pd.Series(h_sid[known]).value_counts()                             # usable actions per search
    c_known = ~np.isnan(c_val)
    ckey = pair_key(c_sid, np.where(c_known, c_val, 0))
    share = counts.reindex(ckey).fillna(0).to_numpy() / totals.reindex(c_sid).to_numpy()   # NaN if no usable actions
    share[~c_known] = np.nan
    return share.astype("float32")

section2 = ex[["sid", "wid"]].copy()
h_sid, c_sid = hist.sid.to_numpy(), ex.sid.to_numpy()
streams2 = {                                                 # name: (which actions, window in days)
    "7d":       (hist.type.isin([TYPE["CLICK"], TYPE["CART"], TYPE["ORD"]]), 7),
    "cart_90d": (hist.type == TYPE["CART"], 90),
    "order_1y": (hist.type == TYPE["ORD"], 365),
}
for sname, (type_mask, days) in streams2.items():
    m = (type_mask & (hist.age_s < days * DAY)).to_numpy()
    for aname, col in ATTRS.items():
        section2[f"{aname}_share_{sname}"] = share_feature(h_sid[m], hist_attr[col].to_numpy()[m], c_sid, cand_attr[col].to_numpy())
    del m
del hist_attr; gc.collect()

# query-matched: last-7-day actions that came from a search similar to today's
h7 = pd.read_parquet(f"{OUT}/history.parquet", columns=["sid", "wid", "type", "qry"], filters=[("age_s", "<", 7 * DAY)])
h7 = h7[h7.type.isin([TYPE["CLICK"], TYPE["CART"], TYPE["ORD"]]) & h7.qry.notna()]
queries = pd.read_parquet(f"{OUT}/searches.parquet", columns=["sid", "query"]).set_index("sid")["query"]   # kept for Sections 4-5

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0

pairs = h7[["sid", "qry"]].drop_duplicates()                 # compare each (search, past query) once
pairs["match"] = [jaccard(tokens(queries.at[s]), tokens(q)) >= 0.5 for s, q in zip(pairs.sid.to_numpy(), pairs.qry.to_numpy())]
matched = h7.merge(pairs[pairs.match], on=["sid", "qry"], how="inner")
m_attr = prods.reindex(matched.wid.to_numpy())
for aname, col in [("cate4", "cate_id_4"), ("shop", "shop_id")]:
    section2[f"qmatch_{aname}_share_7d"] = share_feature(matched.sid.to_numpy(), m_attr[col].to_numpy(), c_sid, cand_attr[col].to_numpy())

save_section(section2, "section2_affinity")
chk = section2[TR]
print("missing (training rows):", [int(chk[c].isna().sum()) for c in section2.columns[2:11]])
print(f"qmatch cate4: n {int(chk.qmatch_cate4_share_7d.notna().sum()):,} mean {chk.qmatch_cate4_share_7d.mean():.4f} | "
      f"qmatch shop: n {int(chk.qmatch_shop_share_7d.notna().sum()):,} mean {chk.qmatch_shop_share_7d.mean():.4f}")
del section2, chk, h7, pairs, matched, m_attr, cand_attr, prods, streams2, type_mask, h_sid, c_sid
_tok_cache.clear(); gc.collect(); ram("section 2")

saved section2_affinity: 11 features x 15,510,012 rows
missing (training rows): [1427365, 1333852, 1333852, 647082, 546458, 546458, 1117337, 1020813, 1020813]
qmatch cate4: n 1,329,629 mean 0.5823 | qmatch shop: n 1,310,902 mean 0.0520
[section 2] RAM in use: 1.9 GB


## Section 3: Popularity from histories (#18–24)

How often each product was clicked (7 days), carted (90 days) and ordered (365 days) across all users' histories. Only actions at least 1 day before the search are counted: all searches took place on the same day, so these actions happened before every search. No labels are used.

Each count is also expressed as a percentile within the search's list. `is_cold` marks products that appear in no user's history from before 17 Oct, the same rule as the counts.

In [ ]:
old_enough = hist.age_s >= DAY                                   # at least 1 day before that user's search
streams3 = {                                                     # name: (action type, window in days)
    "pop_click_7d": (TYPE["CLICK"], 7),
    "pop_cart_90d": (TYPE["CART"], 90),
    "pop_order_1y": (TYPE["ORD"], 365),
}
section3 = ex[["sid", "wid"]].copy()
for name, (t, days) in streams3.items():
    m = old_enough & (hist.type == t) & (hist.age_s < days * DAY)
    counts = hist.wid[m].value_counts()                          # product -> actions by all users in the window
    section3[name] = ex.wid.map(counts).fillna(0).astype("float32")  # never actioned -> 0
    section3[name + "_pct"] = section3.groupby("sid")[name].rank(pct=True).astype("float32")   # position within this list
    del m, counts
section3["is_cold"] = (~ex.wid.isin(hist.wid[old_enough].unique())).astype("int8")   # in no one's history before 17 Oct

save_section(section3, "section3_popularity")
print(section3.iloc[:, 2:].describe().T[["mean", "50%", "max"]].round(3))
print(f"exposures whose product has a 7-day click: {(section3.pop_click_7d > 0).mean():.1%} | cold: {section3.is_cold.mean():.1%}")
del section3, old_enough, streams3; gc.collect(); ram("section 3")

saved section3_popularity: 7 features x 15,510,012 rows
                    mean    50%      max
pop_click_7d       9.226  0.000  10381.0
pop_click_7d_pct   0.506  0.448      1.0
pop_cart_90d      14.505  0.000  30819.0
pop_cart_90d_pct   0.506  0.445      1.0
pop_order_1y       5.967  0.000   4783.0
pop_order_1y_pct   0.506  0.465      1.0
is_cold            0.580  1.000      1.0
exposures whose product has a 7-day click: 28.5% | cold: 58.0%
[section 3] RAM in use: 1.9 GB


## Section 4: Search-level context (diagnostics only)

One row per search:
- days since the user's last action;
- clicks in 7 days, carts in 90 days and orders in 365 days;
- the number of query tokens.

These values are the same for every product in a list, so they cannot change its order and are not model inputs. Steps 3 and 4 use them to break down results by user type.

In [ ]:
section4 = pd.DataFrame({"sid": queries.index.to_numpy()})      # one row per search
section4["days_since_last_action"] = (hist.groupby("sid").age_s.min().reindex(section4.sid).to_numpy() / DAY).astype("float32")
streams4 = {"clicks_7d": (TYPE["CLICK"], 7), "carts_90d": (TYPE["CART"], 90), "orders_1y": (TYPE["ORD"], 365)}
for name, (t, days) in streams4.items():
    m = (hist.type == t) & (hist.age_s < days * DAY)
    section4[name] = hist.sid[m].value_counts().reindex(section4.sid, fill_value=0).to_numpy(dtype="int32")
section4["query_tokens"] = [len(tokens(q)) for q in queries.reindex(section4.sid).to_numpy()]

section4.to_parquet(f"{FEAT}/section4_context.parquet", index=False, compression="zstd")   # one row per search, so not save_section
print(f"saved section4_context: {section4.shape[1] - 1} columns x {len(section4):,} searches")
chk = section4[section4.sid.isin(ex.sid[TR].unique())]          # training searches
print("search 0 (clicks, carts, orders):", section4.loc[section4.sid == 0, ["clicks_7d", "carts_90d", "orders_1y"]].to_numpy()[0].tolist(),
      "| distinct values on training searches: carts", chk.carts_90d.nunique(), "orders", chk.orders_1y.nunique())
print(section4.iloc[:, 1:].describe().T[["mean", "50%", "max"]].round(2))
del section4, chk, m, streams4, hist; _tok_cache.clear(); gc.collect(); ram("section 4")   # no later section needs the history

saved section4_context: 5 columns x 173,831 searches
search 0 (clicks, carts, orders): [49, 126, 7] | distinct values on training searches: carts 1083 orders 725
                         mean    50%      max
days_since_last_action   5.40   0.22   363.96
clicks_7d               47.74  17.00  3575.00
carts_90d               66.30  29.00  3614.00
orders_1y               39.22  15.00  2962.00
query_tokens             2.73   2.00    45.00
[section 4] RAM in use: 1.6 GB


## Section 5: Query × title (#25–28)

- the number of distinct query tokens that appear in the product title;
- that number as a share of the query's tokens;
- the dense rank of the share within the list (1 = best match);
- the title length in tokens.

All four are NaN when the product has no title.


In [ ]:
titles = pd.read_parquet(f"{OUT}/titles.parquet").set_index("wid")["title"]
q_text = queries.reindex(ex.sid.to_numpy()).to_numpy()             # today's query, repeated for every shown product
t_text = titles.reindex(ex.wid.to_numpy()).to_numpy()              # each shown product's title (NaN if none)
del titles; gc.collect()

overlap = np.full(len(ex), np.nan, dtype=np.float32)              # stays NaN if no title or an empty query
q_len = np.zeros(len(ex), dtype=np.float32)
for i, (q, t) in enumerate(zip(q_text, t_text)):
    q_set = tokens(q)
    q_len[i] = len(q_set)
    if isinstance(t, str) and q_set:
        overlap[i] = len(q_set.intersection(t.split(SEP)))       # distinct query tokens found in the title
del q_text, t_text; _tok_cache.clear(); gc.collect()

section5 = ex[["sid", "wid"]].copy()
section5["title_overlap_count"] = overlap
section5["title_overlap_share"] = (overlap / np.where(q_len > 0, q_len, np.nan)).astype("float32")
section5["title_overlap_rank"] = section5.groupby("sid").title_overlap_share.rank(method="dense", ascending=False).astype("float32")
title_len = pd.read_parquet(f"{OUT}/products.parquet", columns=["wid", "title_len"]).set_index("wid")["title_len"]
section5["title_len"] = title_len.reindex(ex.wid.to_numpy()).to_numpy(dtype="float32")

save_section(section5, "section5_text")
chk = section5[TR]
print(f"training rows | overlap count: mean {chk.title_overlap_count.mean():.4f}, missing {int(chk.title_overlap_count.isna().sum()):,} | "
      f"share mean {chk.title_overlap_share.mean():.4f} | rank mean {chk.title_overlap_rank.mean():.4f} | "
      f"title_len mean {chk.title_len.mean():.4f}, missing {int(chk.title_len.isna().sum()):,}")
del section5, chk, overlap, q_len, title_len, queries, q, t, q_set; gc.collect(); ram("section 5")

saved section5_text: 4 features x 15,510,012 rows
training rows | overlap count: mean 1.7827, missing 48,728 | share mean 0.7209 | rank mean 1.5567 | title_len mean 28.8084, missing 48,728
[section 5] RAM in use: 1.8 GB


## Section 6: Same-day outcome statistics (#29–36): excluded from main ML models

Smoothed rates from training searches:
- how often each product was interacted with (clicked, carted or ordered) when shown;
- how often it was ordered when shown;
- the interaction rate of its shop, brand and category;
- how many times it was shown;
- the within-list ranks of the two product rates.

**Out-of-fold.** To stop a row from seeing its own label, training searches are split into 5 folds. Training rows get statistics from the other four folds; validation and test rows get statistics from all training searches. Exposure counts for training rows are scaled by 5/4 to match.

**Smoothing.** Rates are pulled towards the overall rate (`ALPHA` = 20), so products shown only a few times don't get extreme values. Rates are NaN when the shop, brand or category is unknown.

**Why the main models leave these out.** Every search is on 17 Oct, and the data has no time of day. A product's rate therefore counts responses from the whole day, including searches that happened after the one being scored. For example, a 9am search would see a rate built partly from 2pm and 6pm clicks, which JD could not have known at 9am. Out-of-fold stops a row from seeing its own label, but the rates still carry information from later in the day. Sections 1–3 and 5 avoid this: they use the user's own past, or other users' actions from before 17 Oct.

The main models use Sections 1, 2, 3 and 5 (28 features). This file is kept so step 4 can measure what these rates would add if they were available in real time.


In [ ]:
ALPHA, N_FOLDS = 20.0, 5                                           # smoothing strength; number of folds
interacted = (ex.label > LABEL["none"]).to_numpy(np.int8)          # 1 = clicked, carted or ordered
ordered    = (ex.label == LABEL["order"]).to_numpy(np.int8)        # 1 = ordered

train_sids = np.unique(ex.sid[TR])
fold_of = pd.Series(-1, index=train_sids, dtype="int8")
for f, (_, held) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_sids)):
    fold_of.iloc[held] = f                                         # each training search gets a fold 0-4
row_fold = np.full(len(ex), -1, np.int8)                           # -1 = not a training row
row_fold[TR] = fold_of.reindex(ex.sid[TR].to_numpy()).to_numpy()

def label_rates(key, target):
    """Smoothed share of rows with target = 1, per key (product, shop, ...).
    Training rows in fold f use the other folds only; all other rows use every training row."""
    known = ~np.isnan(key)
    out = np.full(len(ex), np.nan, np.float32)
    def fill(stat_rows, rows):
        g = pd.DataFrame({"k": key[stat_rows], "y": target[stat_rows]}).groupby("k").y.agg(["sum", "count"])
        prior = target[stat_rows].mean()                           # overall rate, used to smooth rare keys
        s = g.reindex(key[rows])
        out[rows] = ((s["sum"].fillna(0) + ALPHA * prior) / (s["count"].fillna(0) + ALPHA)).to_numpy()
    fill(TR & known, ~TR & known)                                  # validation / test rows
    for f in range(N_FOLDS):
        fill(TR & known & (row_fold != f), (row_fold == f) & known) # training rows in fold f
    return out

def train_counts(key):
    """How often each key was shown in training searches, same out-of-fold rule.
    Fold counts come from 4 of 5 folds, so they are scaled up by 5/4 to match."""
    out = np.zeros(len(ex), np.float32)
    out[~TR] = pd.Series(key[TR]).value_counts().reindex(key[~TR]).fillna(0).to_numpy()
    for f in range(N_FOLDS):
        held = row_fold == f
        c = pd.Series(key[TR & (row_fold != f)]).value_counts().reindex(key[held]).fillna(0).to_numpy()
        out[held] = c * N_FOLDS / (N_FOLDS - 1)
    return out

wid_key = ex.wid.to_numpy(dtype="float64")                         # float so label_rates can test for NaN
attr = pd.read_parquet(f"{OUT}/products.parquet", columns=["wid", *ATTRS.values()]).set_index("wid").reindex(ex.wid.to_numpy())

section6 = ex[["sid", "wid"]].copy()
section6["item_interact_rate"] = label_rates(wid_key, interacted)
section6["item_order_rate"] = label_rates(wid_key, ordered)
section6["item_exposure_count"] = train_counts(wid_key)
for aname, col in ATTRS.items():
    section6[f"{aname}_interact_rate"] = label_rates(attr[col].to_numpy(dtype="float64"), interacted)
for c in ["item_interact_rate", "item_order_rate"]:
    section6[c + "_rank"] = section6.groupby("sid")[c].rank(method="dense", ascending=False).astype("float32")

save_section(section6, "section6_label_rates")
VA = (ex.split == 1).to_numpy()                                    # validation rows
print(pd.DataFrame({"train (out-of-fold)": section6[TR].iloc[:, 2:].mean(), "validation": section6[VA].iloc[:, 2:].mean(),
                    "missing": section6.iloc[:, 2:].isna().sum()}).round(4))
del section6, attr, wid_key, interacted, ordered, train_sids, fold_of, row_fold; gc.collect(); ram("section 6")

saved section6_label_rates: 8 features x 15,510,012 rows
                         train (out-of-fold)  validation  missing
item_interact_rate                    0.0233      0.0235        0
item_order_rate                       0.0016      0.0016        0
item_exposure_count                   6.1622      6.2137        0
shop_interact_rate                    0.0216      0.0216   219019
brand_interact_rate                   0.0210      0.0210    70338
cate4_interact_rate                   0.0209      0.0210    70338
item_interact_rate_rank               9.9891     11.2708        0
item_order_rate_rank                  5.1923      5.9711        0
[section 6] RAM in use: 1.9 GB


## Section 7: Single-feature checks

Each feature is tested on its own. Every validation list is sorted by that feature alone and scored with NDCG@10, using gains 1 / 3 / 7 for click / cart / order and random tie-breaking. Each feature is tried in both directions and the better one is kept; "-" means lower values rank higher.

The same score is computed on an equal-sized sample of training searches. A feature that scores much higher on training than on validation would be leaking its label. Section 4's features are included to confirm that values shared by a whole list score the same as random. Results are saved to `single_feature_check.csv`.


In [ ]:
GAIN = np.array([0.0, 1.0, 3.0, 7.0])                              # label 0/1/2/3 -> gain 2^label - 1

def ndcg_at_k(sid, label, score, k=10, seed=0):
    """Mean NDCG@k: rank each search's list by `score` (high first, NaN last, ties broken at random)."""
    d = pd.DataFrame({"sid": sid, "label": label, "score": score})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["sid", "score", "tie"], ascending=[True, False, True], na_position="last")
    pos = d.groupby("sid").cumcount().to_numpy()                   # 0 = top of the list
    dcg = pd.Series(np.where(pos < k, GAIN[d.label.to_numpy()] / np.log2(pos + 2), 0.0)).groupby(d.sid.to_numpy()).sum()
    best = d.sort_values(["sid", "label"], ascending=[True, False]) # the perfect order
    bpos = best.groupby("sid").cumcount().to_numpy()
    idcg = pd.Series(np.where(bpos < k, GAIN[best.label.to_numpy()] / np.log2(bpos + 2), 0.0)).groupby(best.sid.to_numpy()).sum()
    ok = idcg > 0                                                  # lists with at least one interaction
    return float((dcg[ok] / idcg[ok]).mean())

sid_all, lab_all = ex.sid.to_numpy(), ex.label.to_numpy()
VA = (ex.split == 1).to_numpy()                                    # validation rows
val_sids, train_sids = np.unique(sid_all[VA]), np.unique(sid_all[TR])
TS = np.isin(sid_all, np.random.default_rng(SEED).choice(train_sids, size=len(val_sids), replace=False))   # as many training searches as validation

def check(section, name, x):
    """Rank every list by one feature alone. Keep the direction (high or low first) that scores better on validation."""
    up = ndcg_at_k(sid_all[VA], lab_all[VA], x[VA])
    down = ndcg_at_k(sid_all[VA], lab_all[VA], -x[VA])
    sign = 1 if up >= down else -1
    train = ndcg_at_k(sid_all[TS], lab_all[TS], sign * x[TS])
    return {"section": section, "feature": name if sign == 1 else "-" + name,
            "valid": max(up, down), "train": train, "missing": float(np.isnan(x[VA]).mean())}

rows = [check("-", "random", np.zeros(len(ex)))]
for sec, file in [(1, "section1_user_product"), (2, "section2_affinity"), (3, "section3_popularity"),
                  (5, "section5_text"), (6, "section6_label_rates")]:
    df = pd.read_parquet(f"{FEAT}/{file}.parquet")
    assert (df.sid.to_numpy() == sid_all).all() and (df.wid.to_numpy() == ex.wid.to_numpy()).all()   # rows line up with ex
    for c in df.columns[2:]:
        rows.append(check(sec, c, df[c].to_numpy(dtype="float64")))
    del df; gc.collect()
section4 = pd.read_parquet(f"{FEAT}/section4_context.parquet").set_index("sid")
for c in section4.columns:
    rows.append(check(4, c, section4[c].reindex(sid_all).to_numpy(dtype="float64")))   # same value across a whole list

checks = pd.DataFrame(rows)
checks.to_csv(f"{FEAT}/single_feature_check.csv", index=False)
print(checks.round(4).to_string(index=False))
del section4; gc.collect(); ram("section 7")

section                    feature  valid  train  missing
      -                     random 0.0832 0.0822   0.0000
      1            item_in_history 0.1737 0.1740   0.0000
      1       item_action_7d_count 0.1590 0.1593   0.0000
      1 -days_since_item_action_7d 0.1591 0.1594   0.9939
      1        item_cart_90d_count 0.1242 0.1244   0.0000
      1        item_order_1y_count 0.0963 0.0964   0.0000
      1  -days_since_item_order_1y 0.0965 0.0965   0.9992
      2              shop_share_7d 0.1497 0.1487   0.1333
      2             brand_share_7d 0.1135 0.1126   0.1245
      2             cate4_share_7d 0.0972 0.0955   0.1245
      2        shop_share_cart_90d 0.1373 0.1363   0.0608
      2       brand_share_cart_90d 0.1030 0.1035   0.0511
      2       cate4_share_cart_90d 0.0935 0.0933   0.0511
      2        shop_share_order_1y 0.1059 0.1064   0.0980
      2       brand_share_order_1y 0.0935 0.0934   0.0889
      2       cate4_share_order_1y 0.0858 0.0849   0.0889
      2      q

### Findings

- **Section 4 scores the same as random.** Values shared by a whole list cannot reorder it, so these stay diagnostics only.
- **No sign that a feature uses training labels.** Training and validation scores are close for every feature, including the out-of-fold rates in Section 6.
- **Section 6 is strong but excluded.** Its exposure count and rates are among the strongest single features. But they use responses from later on the same day (Section 6), so they are not model inputs as they introduce temporal leakage.
- **Popularity is strongest.** Among the model features, history popularity (Section 3) scores highest on its own.
- **Personal history is weaker alone.** The user × product and affinity features (Sections 1–2) are weaker on their own, because most shown products are new to the user.
- **Title overlap is barely above random.** Candidates already tend to match the query: on average, 72% of its tokens appear in the title.
- **`item_order_rate` works best reversed.** Most products have no orders, so a below-average order rate mostly marks a product that was shown often without being ordered. It is likely acting as a stand-in for exposure count.